# CENTRAL-5M Big Data Pipeline Builder & Inspector

This notebook builds, materializes, and validates **CENTRAL-5M**:
- **5,036,579 network flows** across 6 diverse domains (NSL-KDD, UNSW-NB15, BoT-IoT, CIC-IDS2017, CIC-IDS2018, ToN-IoT)
- **19 Canonical Features** standardized via `joint_v2` pipeline preprocessor
- **6 MITRE ATT&CK Classes** aligned with pipeline taxonomy
- **Parquet-First High Speed Storage** (~120 MB total dataset)
- **Canonical Cross-Model Validation (121,199) and Test (143,095) Splits**


In [1]:
import os
import sys
import json
from pathlib import Path
import pandas as pd
import numpy as np

# Locate data-pipeline root
PIPELINE_DIR = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'build_central_5m.py').is_file())
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))

from build_central_5m import build_central_5m, CLASS_NAMES, TACTICS
print(f"Pipeline root located at: {PIPELINE_DIR}")


Pipeline root located at: /Users/bb/Documents/GitHub/CSI_ML_Final/data-pipeline


## 1. Inspect Feature Contract and Dataset Registry
Verify supported domains, features, and table separation schema.

In [2]:
central_5m_dir = PIPELINE_DIR / 'CENTRAL-5M'
with open(central_5m_dir / 'feature_contract.json') as f:
    contract = json.load(f)
with open(central_5m_dir / 'dataset_registry.json') as f:
    registry = json.load(f)

print(f"Pipeline Name: {contract['pipeline_name']} (Contract v{contract['contract_version']})")
print(f"Supported Datasets: {contract['supported_datasets']}")
print(f"Expected Features ({contract['joint_model_profile']['expected_model_feature_count']}):")
for k, v in contract['joint_model_profile']['numeric_features'].items():
    print(f"  - {k:25s}: {v}")
for k, v in contract['joint_model_profile']['categorical_inputs'].items():
    print(f"  - {k:25s}: {v}")


Pipeline Name: CENTRAL-5M (Contract v6)
Supported Datasets: ['NSL-KDD', 'UNSW-NB15', 'CIC-IDS2017', 'BoT-IoT', 'CIC-IDS2018', 'ToN-IoT']
Expected Features (19):
  - log_duration             : log1p(duration)
  - log_bytes_forward        : log1p(bytes_forward)
  - log_bytes_backward       : log1p(bytes_backward)
  - signed_log_bytes_ratio   : log1p(bytes_forward) - log1p(bytes_backward)
  - protocol                 : ['tcp', 'udp', 'icmp', 'other']
  - service_group            : ['dns', 'ftp', 'http', 'https_ssl', 'smtp', 'ssh', 'other']
  - conn_state_group         : ['established', 'no-reply', 'reset', 'other']


## 2. Materialize and Build CENTRAL-5M
Run the automated builder to verify/materialize all Parquet splits, schemas, and audit reports.

In [3]:
# Materialize or verify pipeline
audit_result = build_central_5m(target_dir=central_5m_dir)
print(f"Build Status: {audit_result['status']}")
print(f"Total Multi-domain Records: {audit_result['total_records']:,}")


Building data-pipeline/CENTRAL-5M from final_scale & canonical CENTRAL
Source Directory : /Users/bb/Documents/GitHub/CSI_ML_Final/experiments/scaling_and_domain_shift/data/final_scale
Target Directory : /Users/bb/Documents/GitHub/CSI_ML_Final/data-pipeline/CENTRAL-5M
[1/5] Materializing Harmonized dataset (5.04M records)...
[2/5] Materializing Train split (5.04M records)...
[3/5] Materializing Canonical Validation split (121,199 records)...


[4/5] Materializing Canonical Test split (143,095 records)...


[5/5] Packaging Feature Engineering & Generating Reports...


SUCCESS: CENTRAL-5M pipeline successfully materialized in 6.95s!
Train rows     : 5,036,579
Validation rows: 121,199
Test rows      : 143,095
Build Status: passed
Total Multi-domain Records: 5,300,873


## 3. Verify Splits and Domain Distribution
Inspect the row counts across `train` (5.04M), `validation` (121K), and `test` (143K).

In [4]:
splits_df = pd.read_csv(central_5m_dir / 'output/reports/source_splits.csv')
print(splits_df.to_string(index=False))


     split source_dataset    rows
     train    CIC-IDS2017 2000000
     train        BoT-IoT 1800000
     train    CIC-IDS2018  619346
     train      UNSW-NB15  257673
     train        ToN-IoT  211043
     train        NSL-KDD  148517
validation      UNSW-NB15   49439
validation        BoT-IoT   29912
validation    CIC-IDS2017   28590
validation        NSL-KDD   13258
      test      UNSW-NB15   82332
      test        NSL-KDD   22544
      test        BoT-IoT   20000
      test    CIC-IDS2017   18219


## 4. Inspect Class Balance (MITRE ATT&CK 6-Class Taxonomy)
Check attack distribution across training, validation, and test sets.

In [5]:
train_lbl = pd.read_parquet(central_5m_dir / 'output/splits/train/labels.parquet')
class_counts = train_lbl['attack_category_id'].value_counts().sort_index()

print(f"{'ID':<4} | {'Category Name':<30} | {'Tactic':<35} | {'Flow Count':<12} | {'Percentage'}")
print("-" * 95)
total_train = len(train_lbl)
for cid, count in class_counts.items():
    name = CLASS_NAMES[cid] if 0 <= cid < len(CLASS_NAMES) else 'Unknown'
    tactic = TACTICS[cid] if 0 <= cid < len(TACTICS) else 'Unknown'
    pct = (count / total_train) * 100
    print(f"{cid:<4} | {name:<30} | {tactic:<35} | {count:<12,} | {pct:.2f}%")

norm_count = (train_lbl['label_binary'] == 0).sum()
att_count = (train_lbl['label_binary'] == 1).sum()
print("\nBinary Classification:")
print(f"  - Benign / Normal: {norm_count:,} ({norm_count/total_train*100:.2f}%)")
print(f"  - Attack Flows   : {att_count:,} ({att_count/total_train*100:.2f}%)")


ID   | Category Name                  | Tactic                              | Flow Count   | Percentage
-----------------------------------------------------------------------------------------------
0    | Benign / Normal                | Baseline Operations                 | 2,300,531    | 45.68%
1    | Denial of Service (DoS)        | TA0040: Impact                      | 2,122,048    | 42.13%
2    | Reconnaissance & Probing       | TA0043/TA0007: Recon & Discovery    | 268,792      | 5.34%
3    | Exploits & PrivEsc             | TA0001/TA0004: Initial Access & PrivEsc | 106,872      | 2.12%
4    | Credential Abuse (R2L)         | TA0006: Credential Access           | 130,093      | 2.58%
5    | Malware, C2 & Botnet           | TA0011/TA0008: C2 & Lateral Movement | 108,243      | 2.15%

Binary Classification:
  - Benign / Normal: 2,300,531 (45.68%)
  - Attack Flows   : 2,736,048 (54.32%)


## 5. Sample Feature Space Preview
Inspect sample records of the 19 canonical model features.

In [6]:
train_feat = pd.read_parquet(central_5m_dir / 'output/splits/train/features.parquet')
print(f"Features shape: {train_feat.shape}")
train_feat.head(5)


Features shape: (5036579, 20)


,record_id,log_duration,log_bytes_forward,log_bytes_backward,signed_log_bytes_ratio,proto_tcp,proto_udp,proto_icmp,proto_other,service_dns,service_ftp,service_http,service_https_ssl,service_smtp,service_ssh,service_other,state_established,state_no-reply,state_reset,state_other
0,NSL-KDD:KDDTrain:000000000,-0.69799,0.477092,-0.882694,1.278303,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1,NSL-KDD:KDDTrain:000000001,-0.69799,-0.008192,-0.882694,0.921804,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
2,NSL-KDD:KDDTrain:000000002,-0.69799,-2.012901,-0.882694,-0.550891,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
3,NSL-KDD:KDDTrain:000000003,-0.69799,0.176838,1.645815,-1.600051,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
4,NSL-KDD:KDDTrain:000000004,-0.69799,0.115488,0.813776,-0.770541,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


## 6. Audit Invariants Validation
Verify all strict data integrity invariants.

In [7]:
with open(central_5m_dir / 'output/reports/audit.json') as f:
    audit = json.load(f)

print("Audit Invariants Verification:")
all_passed = True
for inv, passed in audit['invariants'].items():
    status = "PASS" if passed else "FAIL"
    if not passed:
        all_passed = False
    print(f"  [{status}] {inv}")

assert all_passed, "One or more invariants failed!"
print("\nAll CENTRAL-5M pipeline invariants successfully verified.")


Audit Invariants Verification:
  [PASS] train_row_count_ge_5m
  [PASS] validation_row_count_canonical
  [PASS] test_row_count_canonical
  [PASS] feature_count_exact_19
  [PASS] all_features_finite
  [PASS] no_duplicate_train_record_ids

All CENTRAL-5M pipeline invariants successfully verified.
